# Lab 2 · Find the weird transactions — Unsupervised Learning

**Fundamentals of AI** · Same data as Lab 1: [Credit Card Fraud Detection (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

New story: **nobody has labeled anything.** A brand-new fraud pattern appeared — can the machine still find it?

## 1 · Setup & load

**Install libraries** (safe to skip on Colab).

In [ ]:
#@title 0. Install libraries { display-mode: "form" }
# Colab already has these; this makes the notebook work anywhere. -q = quiet.
%pip install -q scikit-learn pandas matplotlib ipywidgets

**Imports + helpers** — theme colors, `scatter2()` for 2D plots, `need()` guard.

In [ ]:
#@title 1a. Theme + helpers { display-mode: "form" }
%matplotlib inline   # charts show inside the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# dark theme colors used by all charts (RED = flagged anomalies)
BG, ACCENT, ACCENT2, TEXT, RED = "#030503", "#00E676", "#3DFF8A", "#FFFFFF", "#FF5252"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "axes.titlecolor": TEXT,
    "grid.color": "#1c3a24", "font.size": 14, "axes.titlesize": 18,
    "axes.labelsize": 15, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "legend.facecolor": "#0a140d", "legend.edgecolor": ACCENT,
    "figure.figsize": (8, 5),
})

def scatter2(Z, c=None, highlight=None, title=""):
    """2D scatter on the dark theme. highlight = boolean mask drawn in red."""
    fig, ax = plt.subplots()
    # main points (optionally colored by cluster), small & semi-transparent
    ax.scatter(Z[:, 0], Z[:, 1], c=c if c is not None else ACCENT2,
               s=4, alpha=0.5, linewidths=0)
    # overlay flagged points in red on top
    if highlight is not None and highlight.any():
        ax.scatter(Z[highlight, 0], Z[highlight, 1], c=RED, s=10, alpha=0.9,
                   linewidths=0, label="flagged")
        ax.legend()
    ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
    ax.set_title(title)
    plt.tight_layout(); plt.show()

def need(*names):
    """Friendly stop if an earlier step hasn't run yet."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise SystemExit(f"✏️ Not ready yet — missing: {', '.join(missing)}. Run earlier cells first (or open 'Show solution').")
print("Theme ready.")

**Load the data** — local CSV first, OpenML fallback; labels are hidden in `answer_key` until §6.

In [ ]:
#@title 1b. Load the data { display-mode: "form" }
import os

# look for a local CSV first (data/creditcard.csv, parent dir, or notebook dir)
csv_path = next((p for p in ["data/creditcard.csv", "../data/creditcard.csv", "creditcard.csv"]
                 if os.path.exists(p)), None)

if csv_path:
    print(f"Loading local file: {csv_path}")
    _df = pd.read_csv(csv_path)
    X_all = _df.drop(columns=["Class"])      # features only
    _y = _df["Class"].astype(int).values     # real labels — kept aside for now
else:
    try:
        from sklearn.datasets import fetch_openml
        # OpenML mirror of the Kaggle dataset — no login needed
        data = fetch_openml(data_id=1597, as_frame=True, parser="auto")
        X_all = data.data if hasattr(data, "data") else data[0]
        _y = (data.target if hasattr(data, "target") else data[1]).astype(int).values
    except Exception as e:
        print("OpenML download failed:", e)
        print("FALLBACK: download creditcard.csv from")
        print("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  (free login),")
        print("put it in a 'data/' folder next to this notebook (or upload via the file panel),")
        print("then re-run this cell.")
        raise SystemExit("Dataset not loaded yet.")
print("Loaded OK.")

# 🔒 THE ANSWER KEY — locked until section 6. Do not peek!
answer_key = _y.copy()   # store real labels under a different name
y_all = None             # the models never see these
print(f"Loaded {X_all.shape[0]:,} transactions. Labels locked in 'answer_key'.")

**Peek at the data** — note there's no `Class` column; labels are locked away.

In [ ]:
#@title 1c. Peek at the data { display-mode: "form" }
print(f"Shape: {X_all.shape[0]:,} rows × {X_all.shape[1]} columns")
X_all.head()   # note: no 'Class' column — labels are locked away

> If 1b fails: download `creditcard.csv` from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), upload it, re-run 1b.

## 2 · Prepare the data

Strip the labels: the machine only sees `X_all`. Different features live on wildly
different scales (`Amount` ~ hundreds, `V1`–`V28` ~ single digits).

### ✏️ Blank #1 — scale the features (1 line)

**Scale the features** — Blank #1: one line so every feature counts equally.

In [ ]:
# ✏️ EDITABLE CELL — students fill the blank
from sklearn.preprocessing import StandardScaler   # rescales each feature to mean 0, std 1

# ✏️ BLANK #1 — YOUR CODE HERE (1 line):
#    scale X_all so every feature counts equally.
#    Hint: StandardScaler().fit_transform( ... )
try:
    X_scaled = StandardScaler().fit_transform(PUT_X_HERE)   # fit on all data (unsupervised — no labels anyway)
except Exception as e:
    print("✏️ Blank #1 not filled — see 'Show solution' below, then re-run.")
    print(f"   ({e})")

**Solution** — run this if you're stuck.

In [ ]:
#@title Show solution { display-mode: "form" }
from sklearn.preprocessing import StandardScaler
# fit_transform learns each feature's mean/std, then rescales — all in one step
X_scaled = StandardScaler().fit_transform(X_all)
print("Scaled: each feature now has mean ≈ 0, std ≈ 1")

<details><summary>💡 Checkpoint — why scale before K-Means?</summary>

K-Means measures **distance** between points. If `Amount` ranges 0–25,000 while `V1`–`V28`
range −10–10, distances are dominated by `Amount` alone — the PCA features barely matter.
Scaling puts everything on equal footing so *all* features vote.</details>

## 3 · Clustering with K-Means

K-Means groups similar transactions into `k` clusters — no labels needed.
Move the sliders and re-run.

**Run K-Means** — move the sliders for k and scaled/raw features.

In [ ]:
#@title 3a. K-Means — knobs
#@param {"type":"slider","min":2,"max":8,"step":1,"label":"k (number of clusters)"}
k = 4
#@param {"type":"boolean","label":"use scaled features? (uncheck to see Amount dominate)"}
use_scaling = True

need("X_all")
from sklearn.cluster import KMeans   # groups similar points into k clusters

# choose which feature set to cluster on
if use_scaling:
    need("X_scaled")          # needs Blank #1
    X_feat = X_scaled
else:
    X_feat = X_all.values     # raw features — Amount will dominate distances

# random sample of up to 50k rows — K-Means on 284k rows would be slow
rng = np.random.RandomState(42)   # fixed seed → same sample every run
idx = rng.choice(len(X_feat), size=min(50_000, len(X_feat)), replace=False)
X_s = X_feat[idx]                 # the sample we actually cluster & plot

kmeans = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_s)
labels_s = kmeans.labels_         # cluster id (0..k-1) for each sampled row
print(f"k={k}, scaling={'ON' if use_scaling else 'OFF'} — fitted on {len(X_s):,} transactions")
print("Cluster sizes (sample):", np.bincount(labels_s))

**Elbow plot** — try k from 2 to 10 and watch inertia drop.

In [ ]:
#@title 3b. Elbow plot — pick k { display-mode: "form" }
need("X_s")
from sklearn.cluster import KMeans

# for each k, fit K-Means and record inertia (total within-cluster distance)
ks = range(2, 11)
inertias = [KMeans(n_clusters=kk, n_init=10, random_state=0).fit(X_s[:20000]).inertia_
            for kk in ks]   # 20k subsample keeps this fast

fig, ax = plt.subplots()
ax.plot(ks, inertias, "o-", color=ACCENT, markersize=8)
ax.set_xlabel("k (clusters)"); ax.set_ylabel("inertia (within-cluster spread)")
ax.set_title("Elbow plot — where does adding clusters stop helping?")
ax.grid(True)
plt.tight_layout(); plt.show()

**Cluster sizes** — how many transactions landed in each cluster.

In [ ]:
#@title 3c. Cluster sizes { display-mode: "form" }
need("labels_s", "k")
fig, ax = plt.subplots()
sizes = np.bincount(labels_s, minlength=k)   # count rows per cluster id
ax.bar(range(k), sizes, color=ACCENT2, width=0.6)
ax.set_xlabel("cluster"); ax.set_ylabel("transactions (sample)")
ax.set_title("How many transactions landed in each cluster")
for i, v in enumerate(sizes):               # print count above each bar
    ax.text(i, v*1.02, f"{v:,}", ha="center", fontsize=13)
plt.tight_layout(); plt.show()

<details><summary>💡 Checkpoint — read the elbow</summary>

**Inertia** = total distance of points to their cluster center. More clusters → lower
inertia, always. The "elbow" is where gains flatten — past that point you're just
splitting noise. There's no single right k; we picked one to see the mechanics.</details>

## 4 · See the data with PCA

30 dimensions is impossible to plot. **PCA** squeezes them into 2 components that keep
as much information as possible — then we can *see* the clusters.
Colored by cluster — **not** by fraud (we don't have labels yet!).

**Project to 2D with PCA** and color by cluster.

In [ ]:
#@title 4. PCA → 2D scatter { display-mode: "form" }
need("X_s", "labels_s")
from sklearn.decomposition import PCA   # squeezes 30 dims → 2 components

pca = PCA(n_components=2, random_state=0).fit(X_s)
Z_s = pca.transform(X_s)                 # project sample into 2D
kept = pca.explained_variance_ratio_.sum() * 100   # % of info retained
print(f"2 components keep {kept:.1f}% of the information in all 30 features")

# color points by cluster id — NOT by fraud (labels still locked)
scatter2(Z_s, c=labels_s, title="Transactions in 2D (PCA), colored by K-Means cluster")

<details><summary>💡 What is PCA doing, in one line?</summary>

It finds the two directions along which the data spreads out the most, then projects
everything onto them. We lose ~some detail (the other 28 dims) but gain a picture.</details>

## 5 · Anomaly detection — Isolation Forest

Different idea: instead of grouping, find the **weird** points. Isolation Forest isolates
random points with random splits — anomalies need fewer splits to be isolated ("easy to
isolate"). Move the slider to flag the top X% most unusual.

**Flag the most unusual** — Isolation Forest scores every transaction; the bottom X% get flagged.

In [ ]:
#@title 5a. Flag the most unusual
#@param {"type":"slider","min":0.5,"max":5,"step":0.5,"label":"flag top X% most unusual"}
flag_pct = 1.0   # slider value: flag the bottom X% most unusual

need("X_s", "X_feat")
from sklearn.ensemble import IsolationForest   # finds outliers via random splits

iforest = IsolationForest(n_estimators=100, random_state=0).fit(X_s)
# score every row (not just sample); LOWER score = more unusual
scores = iforest.score_samples(X_feat)
cutoff = np.quantile(scores, flag_pct / 100)    # threshold: bottom X% are "unusual"
flagged = scores <= cutoff                      # boolean mask over ALL rows
flag_s = flagged[idx]                           # same flags, restricted to plotted sample

print(f"Flagged {flagged.sum():,} transactions ({flag_pct}% of {len(X_feat):,})")

**See the flags on the map** — red = flagged as unusual.

In [ ]:
#@title 5b. Flags on the PCA map { display-mode: "form" }
need("Z_s", "flag_s", "labels_s", "flag_pct")
# same 2D map as before, now with flagged points overlaid in red
scatter2(Z_s, c=labels_s, highlight=flag_s,
         title=f"Red = flagged as unusual (top {flag_pct}%)")

<details><summary>💡 Checkpoint — what does Isolation Forest assume?</summary>

That anomalies are **few and different**: they sit alone, so random splits isolate them
quickly. It knows nothing about fraud — it just finds statistical oddballs. Whether an
oddball is fraud is a question for… the reveal.</details>

## 6 · The reveal

🔓 **Unlocking `answer_key`.** Now we check: of the transactions the machine flagged,
how many were *actually* fraud?

### ✏️ Blank #2 — count flagged frauds (1 line)

**Count the flagged frauds** — Blank #2: one line using `answer_key` and `flagged`.

In [ ]:
# ✏️ EDITABLE CELL — students fill the blank
need("flagged", "answer_key")
n_flagged = int(flagged.sum())        # total rows the machine flagged
total_fraud = int(answer_key.sum())   # all real frauds in the dataset

# ✏️ BLANK #2 — YOUR CODE HERE (1 line):
#    count how many FLAGGED transactions are truly fraud.
#    Hint: answer_key is a 0/1 array aligned with X_all; flagged is a True/False mask.
try:
    fraud_caught = int(ANSWER_HERE)
except Exception as e:
    print("✏️ Blank #2 not filled — see 'Show solution' below, then re-run.")
    print(f"   ({e})")

**Solution** — run this if you're stuck.

In [ ]:
#@title Show solution { display-mode: "form" }
need("flagged", "answer_key")
n_flagged = int(flagged.sum())
total_fraud = int(answer_key.sum())
# answer_key[flagged] keeps only the flagged rows → sum = how many were really fraud
fraud_caught = int(answer_key[flagged].sum())
print(f"Flagged: {n_flagged:,}   Truly fraud among them: {fraud_caught:,}   (of {total_fraud:,} total frauds)")

**Scorecard** — precision, recall, and lift for the flags.

In [ ]:
#@title 6b. Scorecard — precision, recall, lift { display-mode: "form" }
need("fraud_caught", "n_flagged", "total_fraud")
N = len(answer_key)                    # all transactions
precision = fraud_caught / n_flagged   # of flags, share that are real fraud
recall    = fraud_caught / total_fraud # of all frauds, share we caught
base_rate = total_fraud / N            # fraud rate in the whole dataset
lift      = precision / base_rate      # how much richer flags are vs average

print(f"Flagged {n_flagged:,} transactions")
print(f"  truly fraud:        {fraud_caught:,}")
print(f"  precision:          {precision:.3f}   (1 in {1/precision:.0f} flags is real fraud)")
print(f"  recall:             {recall:.3f}   (caught {recall*100:.0f}% of all frauds)")
print(f"  base fraud rate:    {base_rate*100:.3f}%")
print(f"  LIFT:               {lift:.1f}×   (flagged group is {lift:.1f}× richer in fraud)")

**Fraud share per cluster** — does any K-Means cluster hide more fraud than average?

In [ ]:
#@title 6c. Fraud share inside each cluster { display-mode: "form" }
need("kmeans", "X_feat", "answer_key")
# assign every row (not just the sample) to its nearest cluster
clusters_all = kmeans.predict(X_feat)
df_cl = pd.DataFrame({"cluster": clusters_all, "fraud": answer_key})
# per-cluster: mean of 0/1 labels = fraud rate, count = cluster size
share = df_cl.groupby("cluster")["fraud"].agg(["mean", "size"])
share.columns = ["fraud_rate", "count"]
print("K-Means clusters — fraud share vs base rate "
      f"({answer_key.mean()*100:.3f}%):")
display(share.style.format({"fraud_rate": "{:.3%}"}))

**The lesson:** K-Means *groups* similar transactions; Isolation Forest finds *outliers*.
Neither knows what fraud is — the machine only finds "different". **Humans must review the
flags** — that's why precision matters: every false alarm costs analyst time.

## 7 · Stretch — LOF vs Isolation Forest (optional)

**Local Outlier Factor** compares each point to its *neighbors* instead of random splits.
Same question, different algorithm — on a 30k sample that keeps the fraud rate.

**LOF vs Isolation Forest** — same 30k sample, same scorecard, two algorithms.

In [ ]:
#@title 7. LOF vs Isolation Forest (same 30k sample) { display-mode: "form" }
need("answer_key", "X_feat")
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor   # outliers = far from neighbors
from sklearn.ensemble import IsolationForest

# stratified 30k sample — keeps the fraud rate (LOF is slow on full data)
_, idx30 = train_test_split(np.arange(len(X_feat)), test_size=30_000/len(X_feat),
                            stratify=answer_key, random_state=0)
X30, y30 = X_feat[idx30], answer_key[idx30]

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_flags = lof.fit_predict(X30) == -1   # -1 = outlier label from LOF

iso = IsolationForest(n_estimators=100, random_state=0).fit(X30)
# bottom 1% scores = flagged (same trick as section 5)
iso_scores = iso.score_samples(X30)
iso_flags = iso_scores <= np.quantile(iso_scores, 0.01)

base = y30.mean()   # fraud rate in this sample
# same scorecard for both algorithms, side by side
for name, fl in [("Isolation Forest", iso_flags), ("LOF", lof_flags)]:
    prec = y30[fl].mean(); rec = y30[fl].sum() / y30.sum()
    print(f"{name:18s} flagged {fl.sum():>4}  precision {prec:.3f}  recall {rec:.3f}  lift {prec/base:.1f}×")

## 8 · Takeaways

- No labels? **K-Means** finds groups, **Isolation Forest** finds outliers — both unsupervised.
- Neither knows what fraud *is*; they find "different". Humans review the flags.
- **Lift** = how much richer the flagged group is vs the base rate — the number that matters.

🎉 Lab 2 complete.

<details><summary>🔑 Solutions</summary>

- **Blank #1:** `X_scaled = StandardScaler().fit_transform(X_all)`
- **Blank #2:** `fraud_caught = int(answer_key[flagged].sum())`
</details>